In [1]:
from dap_job_quality import PROJECT_DIR

import pandas as pd

In [2]:
data = pd.read_parquet('health_jobs_jq_dimensions.parquet')

In [3]:
job_titles = pd.read_parquet('s3://open-jobs-lake/job_quality/health_social_care/health_jobs_titles_w_soc_codes.parquet')

In [ ]:
len(data)

In [ ]:
data_merged = pd.merge(data, job_titles, on='id', how='left')
len(data_merged)

In [ ]:
len(data_merged['id'].unique())

In [ ]:
data_merged.head()

In [ ]:
data_merged.columns

In [ ]:
result = data_merged.groupby('soc_2020_4_digit')['id'].nunique().reset_index()
result.columns = ['soc_2020_4_digit', 'unique_id_count']
result

In [ ]:
data_merged_deduped = data_merged.drop_duplicates(subset=['description'])
len(data_merged) - len(data_merged_deduped)

In [ ]:
len(data_merged)

In [ ]:
len(data_merged_deduped) / len(data_merged['id'].unique())

In [ ]:
len(data_merged_deduped)/len(data_merged)

In [14]:
contract_df = data_merged[data_merged['subcategory']=='CONTRACT']

In [15]:
contract_df[contract_df['target_phrase']=='zero hour'][['id','description','sentences_split', 'ngrams']].to_csv('zero_hours.csv')

In [ ]:
contract_df[contract_df['description']=='[ Registered Nurse  Join Newcross Healthcare as a Registered Nurse in Fife and the surrounding areas. This role includes part-time and full-time positions in a variety of establishments.  Salary: up to £27.50 per hour (including holiday pay).  Registered Nurse skills and experience   NMC registration 6-months of paid patient-facing UK nursing experience from within the past 3 years Excellent teamwork skills, flexibility and professionalism The right to work in the UK Access to a vehicle would also be beneficial You will need experience in ITU/HDU to care for our service users  Perks of the job   Competitive hourly rates with no zero-hour contracts Generous bonuses if you refer other nurses and carers to Newcross Healthcare Wide range of shifts available across different clinical settings Flexible hours to fit around your lifestyle through our HealthForceGo app Instantly access up to 50% of the value of your completed shifts with Flexi Pay Up to a 30% pay enhancement for working last-minute weekend shifts with Surge Pay Access to Newcross World, our app-based training platform Free ongoing clinical training to further your career Get exclusive deals and discounts on your favourite brands with Perks at Work Free Newcross Healthcare uniform and welcome box when you join Access to NHS registered GPs and mental health support through myHealthPlan Exclusive access to RCNi decision making tools and support with revalidation  About Newcross Healthcare  Newcross Healthcare is one of the UK’s leading independent healthcare providers. We are powered by our fantastic network of 9,000-strong healthcare professionals and bespoke tech that offers a market-leading employee experience. If you’d like to join an  innovative company that continuously invests in the experience of its employees, then join us as a Registered Nurse today!  Please note: To work in a care home in England from 11 November 2021, the post holder is required to be fully vaccinated against Covid-19 unless clinically exempt.  Apply now and you can start within a week! ]']

In [ ]:
len(contract_df)

In [ ]:
contract_df.columns

In [ ]:
contract_df_deduped = contract_df.drop_duplicates(subset=['clean_description'])
len(contract_df) - len(contract_df_deduped)

In [ ]:
result = contract_df.groupby(['soc_2020_4_digit', 'target_phrase'])['id'].nunique().reset_index()
result

In [ ]:
group_cols = ['soc_2020_4_digit', 'target_phrase']

# First, pick one row per 'id' within each ('soc_2020_4_digit', 'target_phrase') group
unique_id_per_group = contract_df.groupby(group_cols + ['sentences_split', 'id'], group_keys=False).apply(lambda x: x.sample(n=1, random_state=42))

# Then, sample at most 5 rows per ('soc_2020_4_digit', 'target_phrase')
eval_sample = unique_id_per_group.groupby(group_cols, group_keys=False).apply(lambda x: x.sample(n=min(4, len(x)), random_state=42))

# Reset index
eval_sample = eval_sample.reset_index(drop=True)

eval_sample

In [ ]:
from dap_job_quality.utils import analysis_utils

eval_sample['classified_contract_type'] = eval_sample['sentences_split'].apply(analysis_utils.classify_contract_type)
eval_sample['classified_contract_type'].value_counts()

In [29]:
eval_sample[['id',
       'SOC', 'soc_2020_4_digit',
       'soc_2020_4_digit_ext', 'Sub-Unit Group',
       'description', 'clean_description',
       'sentences_split', 'ngrams', 'target_phrase', 'cosine_similarity',
       'classified_contract_type']].to_csv('contract_evaluation.csv', index=False)